In [1]:
import pandas as pd
df = pd.read_csv("loan_risk_prediction_dataset.csv")
df["DebtToIncomeRatio"] = df["LoanAmount"] / df["Income"]
df = df.dropna()
df

,Age,Income,LoanAmount,CreditScore,YearsExperience,Gender,Education,City,EmploymentType,LoanApproved,DebtToIncomeRatio
0,56,48353.0,31258.0,675.0,20,Female,High School,Houston,Unemployed,0,0.646454
1,69,57462.0,23262.0,586.0,6,Male,High School,San Francisco,Self-Employed,0,0.404824
2,46,44219.0,26530.0,781.0,26,Male,PhD,Houston,Self-Employed,1,0.599968
4,60,37034.0,27871.0,500.0,19,Female,High School,Chicago,Unemployed,0,0.752579
5,25,47886.0,18106.0,835.0,13,Male,Masters,New York,Salaried,1,0.378106
...,...,...,...,...,...,...,...,...,...,...,...
4992,66,45575.0,32748.0,803.0,39,Female,Masters,San Francisco,Salaried,1,0.718552
4994,21,40705.0,15668.0,396.0,34,Female,Bachelors,Houston,Unemployed,0,0.384916
4996,66,99146.0,9760.0,306.0,14,Male,PhD,New York,Unemployed,0,0.098441
4997,26,58100.0,18230.0,311.0,10,Female,High School,San Francisco,Self-Employed,0,0.313769


In [2]:
data = pd.get_dummies(df)
data.reset_index(drop=True, inplace=True)
data

,Age,Income,LoanAmount,CreditScore,YearsExperience,LoanApproved,DebtToIncomeRatio,Gender_Female,Gender_Male,Education_Bachelors,Education_High School,Education_Masters,Education_PhD,City_Chicago,City_Houston,City_New York,City_San Francisco,EmploymentType_Salaried,EmploymentType_Self-Employed,EmploymentType_Unemployed
0,56,48353.0,31258.0,675.0,20,0,0.646454,True,False,False,True,False,False,False,True,False,False,False,False,True
1,69,57462.0,23262.0,586.0,6,0,0.404824,False,True,False,True,False,False,False,False,False,True,False,True,False
2,46,44219.0,26530.0,781.0,26,1,0.599968,False,True,False,False,False,True,False,True,False,False,False,True,False
3,60,37034.0,27871.0,500.0,19,0,0.752579,True,False,False,True,False,False,True,False,False,False,False,False,True
4,25,47886.0,18106.0,835.0,13,1,0.378106,False,True,False,False,True,False,False,False,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4425,66,45575.0,32748.0,803.0,39,1,0.718552,True,False,False,False,True,False,False,False,False,True,True,False,False
4426,21,40705.0,15668.0,396.0,34,0,0.384916,True,False,True,False,False,False,False,True,False,False,False,False,True
4427,66,99146.0,9760.0,306.0,14,0,0.098441,False,True,False,False,False,True,False,False,True,False,False,False,True
4428,26,58100.0,18230.0,311.0,10,0,0.313769,True,False,False,True,False,False,False,False,False,True,False,True,False


In [3]:
x_data = data.drop('LoanApproved', axis=1).astype(float).values
y_data = data["LoanApproved"].values

In [4]:
from sklearn.model_selection import train_test_split
X_train1, X_test1, y_train1, y_test1 = train_test_split(x_data, y_data, test_size=0.2, random_state=42, stratify=y_data)
len(X_train1), len(X_test1), len(y_train1), len(y_test1)

(3544, 886, 3544, 886)

In [5]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
Xscaled = scaler.fit_transform(X_train1)
Xtscaled = scaler.transform(X_test1)

In [6]:
import tensorflow as tf
X_train = tf.constant(Xscaled, dtype=tf.float32)
X_test = tf.constant(Xtscaled, dtype=tf.float32)
y_train = tf.constant(y_train1, dtype=tf.float32)
y_test = tf.constant(y_test1, dtype=tf.float32)

In [7]:
len(X_train), len(y_train), len(X_test), len(y_test)

(3544, 3544, 886, 886)

In [8]:
from tensorflow.keras import models, layers
input_shape = Xscaled.shape[1]
model = models.Sequential([
    layers.Input(shape=(input_shape,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.05),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss = "binary_crossentropy",
    metrics = ['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    
)
 

In [9]:

class_weights = {0: 1.0, 1: 1.5} 

history = model.fit(
    X_train, y_train,
    epochs=680,
    batch_size=64,
    validation_data=(X_test, y_test),
    class_weight=class_weights,
    verbose=1
)

Epoch 1/680
56/56 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.7398 - loss: 0.6386 - precision: 0.3058 - recall: 0.0748 - val_accuracy: 0.8138 - val_loss: 0.4163 - val_precision: 0.9107 - val_recall: 0.2417
Epoch 2/680
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8476 - loss: 0.4716 - precision: 0.7766 - recall: 0.5036 - val_accuracy: 0.8962 - val_loss: 0.3205 - val_precision: 0.7717 - val_recall: 0.8009
Epoch 3/680
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8809 - loss: 0.3990 - precision: 0.7354 - recall: 0.7791 - val_accuracy: 0.8916 - val_loss: 0.2960 - val_precision: 0.7386 - val_recall: 0.8436
Epoch 4/680
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8984 - loss: 0.3641 - precision: 0.7739 - recall: 0.8088 - val_accuracy: 0.8928 - val_loss: 0.2897 - val_precision: 0.7339 - val_recall: 0.8626
Epoch 5/680
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8956 - loss: 0.3507 - precision: 0.7646 - recall: 0.8100 - val_accuracy: 0.8995 - val_loss: 0.2729

In [15]:
from sklearn.metrics import classification_report, confusion_matrix

predictions = (model.predict(X_test) > 0.5).astype(int)
print(classification_report(y_test, predictions))

28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
              precision    recall  f1-score   support

         0.0       0.96      0.97      0.96       675
         1.0       0.90      0.86      0.88       211

    accuracy                           0.94       886
   macro avg       0.93      0.92      0.92       886
weighted avg       0.94      0.94      0.94       886



In [16]:
model.save('loan_model2.keras')